In [2]:
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from pinecone import ServerlessSpec, Pinecone
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda,
)
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
import os

In [3]:
load_dotenv()

True

In [4]:
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.7)

In [5]:
pc=Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

In [6]:
index_name = "prod-rag"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        serverless=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [7]:
file_path = "D:\ProdRAG\prodRAG\example.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()
print(type(loader))

<class 'langchain_community.document_loaders.pdf.PyPDFLoader'>


In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

In [9]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1370.45it/s]


In [10]:
vectorstore = PineconeVectorStore.from_documents(texts, embedder, index_name=index_name)

In [11]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})


In [12]:
query="What is module 4?"

In [13]:

multi_retriever = MultiQueryRetriever.from_llm(
retriever=retriever,
llm=llm
)

docs = multi_retriever.invoke(query)

In [15]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [16]:
def chunks_docs(docs):
    return [doc.page_content for doc in docs]

In [17]:
rag_prompt = PromptTemplate.from_template(
    template="""You are a helpful assistant.
Use only the provided context to answer the question.
If the answer is not present in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:"""
)

In [ ]:
rag_chain = RunnableParallel(
    {
        "question": RunnablePassthrough(),
        "context": multi_retriever | RunnableLambda(format_docs),
        
    }
)



In [20]:
result=rag_chain | rag_prompt | llm


In [21]:
op=result.invoke("What is summary of module?")


In [22]:
print(op.content)


The provided context describes a course on blockchain technology with several modules. Here's a summary of each module:

- Module 2: Cryptography Fundamentals - Covers the basics of hashing, public and private key encryption, and digital signatures.
- Module 3: Consensus Algorithms - Explains Proof of Work, Proof of Stake, and Byzantine Fault Tolerance.
- Module 4: Smart Contracts and Ethereum - Teaches how to write and deploy smart contracts using Solidity and Remix IDE.
- Module 5: Blockchain and AI Integration - Discusses using blockchain to secure AI model sharing and training data provenance.

There are no course objectives or knowledge provided for Module 1, but based on the repetition of the course objectives, it's likely that Module 1 introduces the fundamentals of blockchain architecture and consensus mechanisms.
